# 🌲 02 — Drishti: Train GBT Delay Predictor (Spark MLlib + MLflow)

This is the **ML core of Drishti**.

**Why GBTRegressor:**
- Native Spark MLlib → distributed training, checks track's "Spark MLlib" technical hook
- Handles mixed categorical + numeric features via StringIndexer + VectorAssembler
- Beats linear models by ~30% on tabular data with non-linear interactions (e.g., fog × peak-hour)
- SHAP works natively on tree-based models → interpretable explanations for the demo
- CPU-native, no GPU needed

**MLflow integration:** every run is logged with params, metrics, the model artifact, and registered to Unity Catalog as `setu_rail.gold.setu_delay_predictor`.

**Target metrics:** MAE ≤ 12 min, RMSE ≤ 22 min.

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

mlflow.set_registry_uri("databricks-uc")   # Unity Catalog model registry
EXPERIMENT = "/Shared/setu-rail-drishti"
mlflow.set_experiment(EXPERIMENT)
print(f"MLflow experiment: {EXPERIMENT}")

In [0]:
# Load the ML-ready feature table
df = spark.table("setu_rail.gold.features_delay_ml") \
    .na.drop(subset=["arrival_delay_min"]) \
    .na.fill({"prev_station_delay": 0.0, "pm25": 60.0, "no2": 25.0, "is_junction": 0})

print(f"Total rows: {df.count():,}")
df.printSchema()

In [0]:
# Time-respecting split: train on first 9 months, test on Q4
train_df = df.filter(col("month") <= 9)
test_df  = df.filter(col("month") >  9)
print(f"Train rows: {train_df.count():,}  |  Test rows: {test_df.count():,}")

In [0]:
# --- Build the ML Pipeline ---
CATEGORICAL = ["train_no", "station_code"]
NUMERIC     = ["dow", "month", "scheduled_hour", "is_peak",
               "pm25", "no2", "is_junction", "prev_station_delay"]
LABEL       = "arrival_delay_min"

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx",
                          handleInvalid="keep") for c in CATEGORICAL]

assembler = VectorAssembler(
    inputCols=[f"{c}_idx" for c in CATEGORICAL] + NUMERIC,
    outputCol="features",
    handleInvalid="skip",
)

# CPU-friendly GBT — moderate depth, modest iterations
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=LABEL,
    maxIter=50,
    maxDepth=6,
    stepSize=0.08,
    maxBins=128,
    seed=42,
)

pipeline = Pipeline(stages=indexers + [assembler, gbt])

In [0]:
# --- Train + log to MLflow ---
with mlflow.start_run(run_name="gbt_v1_baseline") as run:
    model = pipeline.fit(train_df)
    preds = model.transform(test_df)

    mae  = RegressionEvaluator(labelCol=LABEL, metricName="mae").evaluate(preds)
    rmse = RegressionEvaluator(labelCol=LABEL, metricName="rmse").evaluate(preds)
    r2   = RegressionEvaluator(labelCol=LABEL, metricName="r2").evaluate(preds)

    mlflow.log_params({
        "maxIter": 50, "maxDepth": 6, "stepSize": 0.08, "maxBins": 128,
        "train_rows": train_df.count(), "test_rows": test_df.count(),
        "features": ",".join(CATEGORICAL + NUMERIC),
    })
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})

    # Log model with signature for clean loading later
    sig = infer_signature(test_df.limit(50).toPandas(),
                          preds.select("prediction").limit(50).toPandas())
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="setu_gbt_model",
        signature=sig,
        registered_model_name="setu_rail.gold.setu_delay_predictor",
    )

    print(f"\n✅ MAE:  {mae:.2f} min")
    print(f"✅ RMSE: {rmse:.2f} min")
    print(f"✅ R²:   {r2:.3f}")
    print(f"   Run ID: {run.info.run_id}")

In [0]:
# --- Baseline comparison: 'always predict mean' — to prove the model adds value ---
mean_delay = train_df.agg({"arrival_delay_min": "avg"}).collect()[0][0]
from pyspark.sql.functions import lit, abs as abs_, pow as pow_

base_preds = test_df.withColumn("prediction", lit(mean_delay))
baseline_mae = base_preds.withColumn("e", abs_(col("arrival_delay_min") - col("prediction"))) \
                         .agg({"e": "avg"}).collect()[0][0]

improvement_pct = 100 * (baseline_mae - mae) / baseline_mae
print(f"Baseline (predict mean) MAE: {baseline_mae:.2f} min")
print(f"GBT model MAE:              {mae:.2f} min")
print(f"🎯 Improvement over baseline: {improvement_pct:.1f}%")

In [0]:
# --- Persist predictions to gold for dashboard ---
preds.select(
    "train_no", "station_code", "run_date", "scheduled_hour",
    col("arrival_delay_min").alias("actual_delay_min"),
    col("prediction").alias("predicted_delay_min"),
    "pm25", "no2",
).write.format("delta").mode("overwrite") \
 .saveAsTable("setu_rail.gold.predictions_daily")

print("✅ gold.predictions_daily written — dashboard-ready")

In [0]:
# --- Promote model to Production alias in UC Registry ---
from mlflow import MlflowClient
client = MlflowClient()
versions = client.search_model_versions("name='setu_rail.gold.setu_delay_predictor'")
latest = max(versions, key=lambda v: int(v.version))
client.set_registered_model_alias(
    name="setu_rail.gold.setu_delay_predictor",
    alias="production",
    version=latest.version,
)
print(f"✅ Aliased setu_delay_predictor v{latest.version} as @production")

✅ **Next:** `03_shap_explainability.ipynb`